# RAG 기초(RAG Basics)

이 노트북은 가장 단순한 형태의 검색 증강 생성(Retrieval-Augmented Generation, RAG) 파이프라인을 처음부터 끝까지 따라가며 이해하는 것을 목표로 한다. 먼저 원문 문서를 불러와 어떻게 작은 청크(chunk)로 나누는지 보고, 그다음 검색 표현 방식이 결과를 어떻게 바꾸는지 비교한다. 마지막에는 baseline RAG가 어느 지점까지는 잘 작동하지만, 왜 곧바로 한계에 부딪히는지도 확인한다.

## 학습 목표
- RAG가 일반적인 생성형 답변과 무엇이 다른지 이해한다.
- 청킹(chunking), 검색(retrieval), 답변 생성(answer generation)이 어떻게 연결되는지 본다.
- TF-IDF 기반 baseline이 왜 빠르고 설명 가능하지만, planning·tool use·근거 검증(grounding verification)에는 약한지 설명할 수 있게 된다.
- 다음 노트북인 agentic workflow로 왜 넘어가야 하는지 자연스럽게 이해한다.


## 개념 설명

이 첫 코드 셀은 노트북이 어떤 Python 환경에서 실행되는지 확인하고, 현재 세션이 `src/` 패키지를 제대로 읽을 수 있게 경로를 정리한다. 학습 노트북에서는 실행 환경이 조금만 달라도 import 실패나 커널 혼선이 자주 생기므로, 가장 먼저 `sys.executable`과 runtime 설정을 확인하는 습관이 중요하다.

- **목적**: 노트북이 올바른 `uv` 가상환경과 프로젝트 루트를 사용하고 있는지 검증한다.
- **핵심 로직**: `Path.cwd()`로 현재 디렉터리를 잡고, `src/`가 없으면 한 단계 위로 올라가 프로젝트 루트를 찾는다. 그런 다음 `sys.path`에 루트를 넣어 `src.config`를 import할 수 있게 만든다.
- **주요 파라미터/변수**:
  - `ROOT`: 노트북 실행 기준이 되는 프로젝트 루트 경로이다.
  - `sys.path`: Python이 모듈을 찾는 검색 경로 목록이다.
  - `RuntimeConfig.auto_detect()`: 현재 장치가 `cpu`, `mps`, `cuda` 중 무엇을 쓸 수 있는지 자동 감지한 설정 객체를 반환한다.

예를 들어 아래 코드에서:
- `if not (ROOT / 'src').exists()`: 현재 작업 디렉터리 바로 아래에 `src`가 없으면 부모 디렉터리를 프로젝트 루트로 보겠다는 뜻이다.
- `print(sys.executable)`: 지금 연결된 Jupyter kernel이 어떤 Python 실행 파일을 쓰는지 보여준다.
- `print(RuntimeConfig.auto_detect())`: 검색 백엔드와 디바이스 설정이 어떤 값으로 잡혔는지 빠르게 점검하게 해준다.

💡 면접 포인트: 실무에서는 모델 품질보다 먼저 "이 노트북이 어느 환경에서 돌고 있나"를 명확히 하는 것이 재현성을 좌우한다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## RAG란 무엇인가

RAG는 먼저 문서를 검색한 뒤, 그 검색 결과를 근거로 답변을 만드는 구조이다. 일반적인 검색엔진은 관련 문서를 찾는 데서 끝나는 경우가 많고, 일반적인 생성 모델은 내부 파라미터에 저장된 지식만으로 답하려는 경향이 있다. RAG는 이 둘을 연결해서, **질문과 관련 있는 외부 문서 조각을 먼저 찾고 그 조각을 바탕으로 답을 생성**한다는 점이 핵심이다.

이 셀에서는 현재 프로젝트에 포함된 원문 문서(raw documents)를 불러와 데이터셋의 범위를 먼저 파악한다. 어떤 문서가 들어 있는지 모르면, 나중에 retrieval이 성공했는지 실패했는지도 판단할 수 없다.

- **목적**: 답변의 근거가 되는 문서 집합을 확인한다.
- **핵심 로직**: `load_documents()`가 `data/raw/` 아래 파일을 읽어 `doc_id`, `source`, `text`를 가진 문서 리스트를 반환하고, 이를 `DataFrame`으로 정리해 사람이 읽기 쉽게 만든다.
- **주요 파라미터/변수**:
  - `documents`: 로드된 전체 문서 목록이다.
  - `document_frame`: 문서 ID, 파일명, 길이를 요약한 표이다.
  - `characters`: 문서 길이를 문자 수 기준으로 대략 확인하기 위한 컬럼이다.

이 결과를 볼 때는 문서 수가 너무 적지는 않은지, 특정 문서만 유난히 긴지, 질문에 답할 만한 주제가 실제로 들어 있는지를 같이 확인해보자. 이후의 retrieval 성능은 이 원문 집합의 품질을 절대 넘지 못한다.


In [ ]:
import pandas as pd

from src.ingestion import load_documents

documents = load_documents()
document_frame = pd.DataFrame(
    [
        {'doc_id': item['doc_id'], 'source': item['source'], 'characters': len(item['text'])}
        for item in documents
    ]
)
document_frame

## 문서 청킹(chunking) 이해하기

검색은 보통 문서 전체를 한 번에 비교하지 않고, 더 작은 청크(chunk) 단위로 비교한다. 이 프로젝트는 문장(sentence) 기반 청킹을 사용한다. 문장 기반 청킹은 사람이 읽었을 때 의미가 자연스럽고, 인접 문장을 묶어 문맥을 보전하기 쉽다는 장점이 있다. 반면 고정 길이 토큰/문자 청킹은 구현이 단순하고 벡터 인덱싱이 쉬울 수 있지만, 문장 중간이 잘려 의미가 깨질 위험이 있다.

이 셀은 문서를 청크로 바꾼 뒤, 실제로 검색 대상이 어떤 단위가 되는지 눈으로 보여준다.

- **목적**: retrieval의 실제 입력 단위가 문서 전체가 아니라 청크라는 점을 확인한다.
- **핵심 로직**: `ingest_documents(persist=False)`가 원문을 읽고 sentence-aware chunking을 수행한 뒤, 메모리 안에서만 청크 리스트를 반환한다.
- **주요 파라미터/변수**:
  - `persist=False`: 청크를 파일로 저장하지 않고 이번 세션 메모리에서만 유지하겠다는 뜻이다. 노트북 실험에서는 빠른 반복을 위해 자주 사용한다.
  - `chunk_frame`: 생성된 청크를 표 형태로 살펴보기 위한 데이터프레임이다.
  - `chunk_id`, `source`, `text`: 각 청크가 어느 문서에서 왔고 어떤 텍스트를 담는지 보여준다.

예를 들어 아래 코드에서:
- `chunks = ingest_documents(persist=False)`: 전체 문서를 검색 가능한 조각으로 바꾼다.
- `chunk_frame[['doc_id', 'chunk_id', 'source', 'text']].head(10)`: 처음 10개 청크를 보면서 청킹이 너무 잘게 쪼개지지 않았는지, 혹은 너무 길지 않은지 확인한다.

청크가 너무 짧으면 근거가 분산되고, 너무 길면 관련 없는 문장이 섞여 검색 점수가 흐려진다. RAG 품질의 상당 부분은 여기서 결정된다.


In [ ]:
from src.ingestion import ingest_documents

chunks = ingest_documents(persist=False)
chunk_frame = pd.DataFrame(chunks)
chunk_frame[['doc_id', 'chunk_id', 'source', 'text']].head(10)

## 검색 표현 방식: TF-IDF와 dense retrieval

이제 청크를 실제 검색 인덱스로 바꾼다. 여기서는 두 가지 경로를 비교한다. 첫 번째는 전통적인 sparse 표현인 TF-IDF이고, 두 번째는 가능할 때 사용하는 dense/FAISS 경로이다. TF-IDF는 특정 단어가 문서 안에서 얼마나 특징적인지를 수치화하므로 빠르고 설명 가능하다. 반면 dense embedding은 단어가 정확히 일치하지 않아도 의미적으로 비슷한 문장을 잡아낼 가능성이 높다.

- **목적**: 같은 문서 집합이라도 어떤 검색 표현을 쓰느냐에 따라 retriever 객체가 달라진다는 점을 확인한다.
- **핵심 로직**: `build_demo_index()`가 청크를 만든 뒤, `backend='tfidf'`이면 기존 `HybridRetriever`를, `backend='faiss'`이면 dense retrieval 경로를 준비한다. 환경이 dense 경로를 지원하지 않으면 내부적으로 안전하게 폴백한다.
- **주요 파라미터/변수**:
  - `backend='tfidf'`: 단어 빈도 기반 sparse 검색을 사용한다.
  - `backend='faiss'`: dense index를 시도한다. GPU/optional dependency 상태에 따라 실제 클래스는 달라질 수 있다.
  - `retriever_summary`: 어떤 백엔드가 어떤 클래스로 초기화되었는지 비교하는 표이다.

예를 들어:
- `build_demo_index(persist=False, backend='tfidf')`: baseline에 쓰기 좋은 가볍고 안정적인 인덱스를 만든다.
- `build_demo_index(persist=False, backend='faiss')`: 의미 기반 검색을 위한 확장 경로를 시도한다.

💡 면접 포인트: TF-IDF는 "왜 이 문서가 나왔는지" 설명하기 쉽고, dense retrieval은 "단어가 달라도 비슷한 뜻"을 잡기 좋다. 실제 시스템은 둘 중 하나만 고집하기보다 용도에 맞게 선택하거나 혼합한다.


In [ ]:
from src.ingestion import build_demo_index

tfidf_retriever = build_demo_index(persist=False, backend='tfidf')
faiss_retriever = build_demo_index(persist=False, backend='faiss')
retriever_summary = pd.DataFrame(
    [
        {'backend': 'tfidf', 'class_name': type(tfidf_retriever).__name__},
        {'backend': 'faiss', 'class_name': type(faiss_retriever).__name__},
    ]
)
retriever_summary

## 구현: 실제 검색 결과 읽기

이 셀은 같은 질문을 두 검색기(retriever)에 넣었을 때 어떤 상위 결과(top-k)가 나오는지 직접 비교한다. 검색 시스템을 공부할 때는 최종 답변보다 먼저 **retrieval 자체가 맞는 근거를 가져오는지**를 확인하는 습관이 중요하다. 답이 틀렸을 때 원인이 생성 단계인지, 검색 단계인지 분리할 수 있기 때문이다.

- **목적**: 질문 하나를 기준으로 TF-IDF와 FAISS 경로가 어떤 청크를 상위에 올리는지 비교한다.
- **핵심 로직**: `retriever.search(query, top_k=4)`가 쿼리와 가장 관련 높은 청크 4개를 반환한다. 각 결과에는 `chunk_id`, `source`, `score`, `text`가 포함된다.
- **주요 파라미터/변수**:
  - `retrieval_query`: 검색 품질을 시험할 질문이다.
  - `top_k=4`: 상위 4개 청크만 가져오겠다는 뜻이다. 너무 작으면 근거를 놓치고, 너무 크면 노이즈가 섞일 수 있다.
  - `score`: retriever가 계산한 관련성 점수이다. 점수 절대값보다 **상대적 순위**를 함께 봐야 한다.

예를 들어 아래 코드에서:
- `tfidf_retriever.search(retrieval_query, top_k=4)`: 단어 기반 일치에 강한 검색 결과를 반환한다.
- `faiss_retriever.search(retrieval_query, top_k=4)`: 의미적으로 가까운 문장을 더 잘 잡을 수 있는 경로를 보여준다.

결과를 읽을 때는 `source`가 질문과 맞는 문서를 가리키는지, `text` 안에 실제 답의 단서가 들어 있는지, 상위 1~2개 결과에 이미 핵심 문장이 보이는지를 확인해보자. 보통 점수가 0.5 이상이면 꽤 강한 힌트일 수 있고, 0.3 이하 영역은 문서 규모에 따라 노이즈일 가능성도 염두에 둔다.


In [ ]:
retrieval_query = 'What are the goals of the workspace policy refresh?'
tfidf_results = pd.DataFrame(tfidf_retriever.search(retrieval_query, top_k=4))
faiss_results = pd.DataFrame(faiss_retriever.search(retrieval_query, top_k=4))

print('TF-IDF top results')
display(tfidf_results[['chunk_id', 'source', 'score', 'text']])
print('FAISS path top results')
display(faiss_results[['chunk_id', 'source', 'score', 'text']] if not faiss_results.empty else faiss_results)

## baseline 답변 생성(Simple QA)

이제 검색된 문서를 바탕으로 가장 단순한 baseline 답변을 만든다. 이 baseline은 planning도 없고, tool use도 없고, 근거 검증(grounding verification)도 강하게 하지 않는다. 대신 구조가 매우 단순해서 "RAG의 최소 동작 단위"를 설명하기에 좋다.

- **목적**: retrieval 결과를 이용해 baseline RAG가 어떤 형태의 답변을 만드는지 확인한다.
- **핵심 로직**: `run_baseline_rag(question, retriever=...)`가 질문을 검색기에 보내고, 상위 문서 조각을 모아 간단한 합성 답변을 만든 뒤 `final_status`, `final_answer`, `retrieved_docs`를 담은 상태를 반환한다.
- **주요 파라미터/변수**:
  - `question`: baseline이 답해야 할 사용자 질문이다.
  - `retriever=tfidf_retriever`: 어떤 검색기를 쓸지 명시한다. 여기서는 baseline 비교를 위해 TF-IDF 경로를 고정한다.
  - `retrieved_sources`: 답변 생성에 실제로 사용된 근거 문서 파일명 목록이다.

예를 들어:
- `run_baseline_rag(..., retriever=tfidf_retriever)`: 검색과 간단한 답변 생성을 하나의 함수로 묶어 실행한다.
- `baseline_result['retrieved_docs']`: 답변이 어떤 문서 조각에 기대고 있는지 역으로 확인할 수 있다.

이 단계에서 중요한 질문은 "답이 그럴듯한가"보다도 "이 답이 어떤 근거에서 왔는가"이다. baseline은 빠르지만, 근거가 약한 상황에서도 비교적 쉽게 자신 있게 말할 수 있다는 한계가 있다.


In [ ]:
from src.workflow import run_baseline_rag

baseline_result = run_baseline_rag(
    'What are the main goals of the workspace policy refresh?',
    retriever=tfidf_retriever,
)

pd.Series(
    {
        'final_status': baseline_result['final_status'],
        'final_answer': baseline_result['final_answer'],
        'retrieved_sources': [item['source'] for item in baseline_result['retrieved_docs']],
    }
)

## 실험: 쉬운 질문과 어려운 질문 비교

baseline의 강점과 약점은 질문 유형을 바꿔보면 더 선명하게 드러난다. 이 셀은 비교적 직접 답할 수 있는 질문과 계산/추론이 필요한 질문을 나란히 넣어 본다. 같은 RAG 구조라도 질문 성격이 달라지면 필요한 처리 단계가 달라진다는 점을 체감하는 것이 목적이다.

- **목적**: baseline이 어떤 질문에는 잘 답하고, 어떤 질문에는 구조적으로 약한지 비교한다.
- **핵심 로직**: `experiment_questions`에 두 질문을 넣고, 각 질문에 대해 `run_baseline_rag()`를 반복 실행한 뒤 결과를 표로 정리한다.
- **주요 파라미터/변수**:
  - `experiment_questions`: 서로 난이도가 다른 질문 목록이다.
  - `experiment_rows`: 각 실행 결과를 누적하는 리스트이다.
  - `retrieved_sources`: 어떤 문서를 근거로 잡았는지 보여주는 요약 필드이다.

아래 코드에서:
- `'When does the organization-wide rollout begin?'`: 문서에 날짜가 직접 있으면 baseline도 비교적 쉽게 답할 수 있다.
- `'How many days are in the pilot window?'`: 날짜 차이 계산이 필요하므로 retrieval만으로는 충분하지 않을 수 있다.

출력 표를 볼 때는 `final_status`가 모두 answered여도 안심하면 안 된다. 실제로는 두 번째 질문처럼 tool이나 planning이 있어야 더 안정적으로 답할 수 있는 경우가 많다.


In [ ]:
experiment_questions = [
    'When does the organization-wide rollout begin?',
    'How many days are in the pilot window?',
]
experiment_rows = []
for question in experiment_questions:
    result = run_baseline_rag(question, retriever=tfidf_retriever)
    experiment_rows.append(
        {
            'question': question,
            'final_status': result['final_status'],
            'retrieved_sources': ', '.join(sorted({doc['source'] for doc in result['retrieved_docs']})),
            'answer': result['final_answer'],
        }
    )

pd.DataFrame(experiment_rows)

## 결과 해석 가이드

이 마지막 분석 셀은 baseline과 dense retrieval 확장 경로를 한 문장으로 정리해준다. 여기서 중요한 것은 어느 쪽이 무조건 더 좋다는 결론이 아니라, **각 접근이 어떤 문제를 풀기 위해 존재하는지**를 구분해서 읽는 것이다.

- **목적**: baseline 설계의 장단점을 명시적으로 정리한다.
- **핵심 로직**: `analysis_frame`에 각 백엔드의 강점과 한계를 정리해 표로 보여준다.
- **주요 파라미터/변수**:
  - `backend`: 비교 대상 시스템 이름이다.
  - `strength`: 이 접근을 선택할 때 기대할 수 있는 이점이다.
  - `limitation`: 다음 단계 설계가 필요한 이유를 보여주는 제약 조건이다.

이 표를 읽을 때는 다음처럼 해석하면 된다.
- `Fast, deterministic, easy to inspect`: baseline은 재현성과 설명 가능성이 높다. 데모, 교육, 디버깅에 좋다.
- `No planning, tool use, or abstention logic`: 하지만 질문 유형을 구분하지 못하고, 계산 도구도 못 쓰며, 근거가 약할 때 멈추는 전략도 부족하다.
- `Dense similarity can surface semantically close evidence`: dense retrieval은 의미적으로 비슷한 문장을 찾는 데 유리하다.
- `Needs extra dependencies and more setup`: 대신 환경 의존성과 운영 복잡도가 올라간다.

이 한계가 바로 다음 노트북에서 agentic workflow가 필요한 이유다. 단순 RAG만으로는 "무엇을 먼저 해야 하는가"를 판단하기 어렵다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'backend': 'TF-IDF baseline',
            'strength': 'Fast, deterministic, easy to inspect',
            'limitation': 'No planning, tool use, or abstention logic',
        },
        {
            'backend': 'Optional FAISS path',
            'strength': 'Dense similarity can surface semantically close evidence',
            'limitation': 'Needs extra dependencies and more setup',
        },
    ]
)
analysis_frame

## 핵심 정리

이 노트북을 통해 baseline RAG의 가장 작은 동작 단위를 확인했다. 문서를 불러오고, 청크로 나누고, 검색기로 관련 청크를 찾은 다음, 그 근거를 바탕으로 답변을 만드는 흐름 자체는 생각보다 단순하다.

다만 실험 결과를 보면 baseline은 질문 유형을 구분하지 못하고, 계산이나 날짜 추론처럼 추가 도구가 필요한 작업에도 동일한 방식으로 접근한다. 또한 답변이 충분히 근거 있는지 검증하는 절차도 약하다.

💡 면접 포인트: "baseline RAG는 retrieval과 synthesis를 빠르게 보여주기에는 좋지만, 실제 서비스 수준의 신뢰성을 위해서는 planning, tool use, grounding verification, abstention이 추가로 필요하다"고 설명하면 좋다. 다음 노트북은 바로 그 확장인 stateful agent workflow를 다룬다.
